# Split KSE daily files into date-range sheets

This notebook splits the KSE CSV files (kse100 and kse30) into multiple Excel sheets.
Each sheet contains data from the start up to (and including) a cutoff date.
Outputs are written to the `work/3_merge_data/recomposition_data` folder and a processing log CSV is created.

In [1]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# --- Configuration ---
# Date cutoffs provided by the user (strings).
date_cutoffs = [
    '2021-10-01',
    '2021-04-01',
    '2022-10-03',
    '2023-04-03',
    '2023-10-01',
    '2024-04-01',
    '2024-10-01',
    '2025-04-01',
]
# Convert to datetimes and sort ascending (makes slicing easier)
cutoffs = sorted(pd.to_datetime(date_cutoffs))

# Input CSV folder (existing) and output folder
input_folder = Path(r'e:/Codes/Python/indexfundproject/work/3_merge_data/csvs')
output_folder = Path(r'e:/Codes/Python/indexfundproject/work/3_merge_data/recomposition_data')
output_folder.mkdir(parents=True, exist_ok=True)

# Files to process (CSV names in the csvs folder)
files = ['kse100_daily_data.csv', 'kse30_daily_data.csv']

print('Configuration:')
print('-', 'input_folder =', input_folder)
print('-', 'output_folder =', output_folder)
print('-', 'cutoffs =', [d.strftime('%Y-%m-%d') for d in cutoffs])

Configuration:
- input_folder = e:\Codes\Python\indexfundproject\work\3_merge_data\csvs
- output_folder = e:\Codes\Python\indexfundproject\work\3_merge_data\recomposition_data
- cutoffs = ['2021-04-01', '2021-10-01', '2022-10-03', '2023-04-03', '2023-10-01', '2024-04-01', '2024-10-01', '2025-04-01']


In [2]:
def split_and_write(file_name, input_folder, output_folder, cutoffs):
    """
    Read a CSV file with a 'Date' column, split into segments from start up to each cutoff,
    and write an Excel file with one sheet per cutoff. Returns a small log.
    """
    in_path = input_folder / file_name
    if not in_path.exists():
        raise FileNotFoundError(f'Input file not found: {in_path}')

    df = pd.read_csv(in_path, parse_dates=['Date'])
    if df.empty:
        print(f'Warning: {file_name} is empty')

    df = df.sort_values('Date')
    # Ensure Date is datetime
    df['Date'] = pd.to_datetime(df['Date'])

    out_name = file_name.replace('.csv', '') + '_recomposition.xlsx'
    out_path = output_folder / out_name

    log_records = []
    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        for cutoff in cutoffs:
            mask = df['Date'] <= cutoff
            seg = df[mask].copy()
            sheet_name = f'to_{cutoff.strftime('%Y-%m-%d')}'
            # Excel sheet name limit is 31 chars
            seg.to_excel(writer, sheet_name=sheet_name[:31], index=False)
            log_records.append({
                'input_file': file_name,
                'output_file': out_name,
                'sheet': sheet_name[:31],
                'rows': len(seg)
            })
    return log_records

In [3]:
# Run the split for each file and collect a processing log
processing_log = []
for f in files:
    try:
        log = split_and_write(f, input_folder, output_folder, cutoffs)
        processing_log.extend(log)
        print(f'Finished {f} -> {len(log)} sheets, output:', (output_folder / (f.replace('.csv','') + '_recomposition.xlsx')).name)
    except Exception as e:
        print('Error processing', f, ':', e)

# Save processing log as CSV
if processing_log:
    log_df = pd.DataFrame(processing_log)
    log_path = output_folder / 'processing_log.csv'
    log_df.to_csv(log_path, index=False)
    print('Saved processing log to', log_path)
else:
    print('No processing records to save')

KeyboardInterrupt: 